# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
import os
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
logging.info(f'Path [{os.environ["PATH"]}]')

# Test pytorch GPU config
import torch
cuda_test = torch.cuda.is_available()
logging.info(f"✅ Torch CUDA available: {cuda_test}")
device_name = torch.cuda.get_device_name(0)
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"🖥️ Device Name: {device_name} | Device reference: {torch_device.type}")

### machine learning (scikit-learn)
import math
import pprint
import numpy as np
import pandas as pd
import pprint
from typing import cast
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings(
    "ignore",
    message="mtime may not be reliable on this filesystem, falling back to numerical ordering"
)
from transformers import set_seed, Trainer  # type: ignore
from tsfm_public import (
    TimeSeriesForecastingPipeline,
)
from tsfm_public.toolkit.time_series_preprocessor import get_datasets, prepare_data_splits
from tsfm_public.toolkit.visualization import plot_predictions
from tsfm_public.toolkit.service_util import save_deployment_package

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline


## 1.2 Project Specific imports

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.preprocessing_project_specific as pps
import smartcheck.deep_learning_project_specific as dlps
import smartcheck.modeling_project_specific as mps

# 2. Loading and Preprocessing

## 2.1 Loading data

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Data Refactoring pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    # "date_et_heure_de_comptage",
    "date_et_heure_de_comptage_local",
    # "date_et_heure_de_comptage_utc",
    "orientation_compteur",
    # "latitude",
    # "longitude",
    # "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    # "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("convert_datetime", pps.DatetimePreprocessingTransformer(timestamp_col="date_et_heure_de_comptage",
                                                                   for_sarimax=True)),
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

# 3. Modeling Training and Prediction

## 3.1 Contextual variables

#### For all experiments (common)

In [ ]:
OUT_DIR = "ttm_results.model" # model runtime and training data archiving/export.
# Model inner parameters 
context_length = 512 # max possible for this model
prediction_length = 96 # max possible for this model
fcm_context_length = 96 # 1 lag = 1 hour
# Parameters for the training
learning_rate: float = 0.0002
num_epochs: int = 200
patience: int = 10
batch_size: int = 128 # à ajuster par rapport à la mémoire dédiée GPU
# Parameters for the preprocessing
fewshot_fraction = 1 # fasten training by returning a percent original train dataset
set_seed(42)
split_config = {"train": 0.6, "test": 0.25}
timestamp_column = "date_et_heure_de_comptage_local"
target_columns = ["comptage_horaire"]
column_specifiers = {
    "timestamp_column": timestamp_column,
    "id_columns": [
        "nom_du_site_de_comptage",
        "orientation_compteur",
    ],
    "target_columns": target_columns,
    "categorical_columns": [
        "vacances_scolaires",
        "weather_code_wmo_code_category",
    ],
    "control_columns": [
    ],
    "observable_columns": [
        "jour_ferie",
        "vacances_scolaires",
        "temperature_2m_c",
        "rain_mm",
        "snowfall_cm",
        "weather_code_wmo_code_category",
    ],
}

#### For each specific experiments

In [ ]:
dict_compteurs = {
    # "experiment_1": {
    #     "name": "Subset_9Counters_fcm168",
    #     "counters_dict": {
    #         ('Totem 73 boulevard de Sébastopol', 'S-N'): "Sébastopol_S-N",
    #         ('Totem 73 boulevard de Sébastopol', 'N-S'): "Sébastopol_N-S",
    #         ('102 boulevard de Magenta', 'SE-NO'): "Magenta_SE-NO",
    #         ('Pont de Bercy', 'NE-SO'): "Bercy_NE-SO",
    #         ('Pont de Bercy', 'SO-NE'): "Bercy_SO-NO",
    #         ('135 avenue Daumesnil', 'SE-NO'): "Daumesnil_SE-NO",
    #         ("180 avenue d'Italie", 'N-S'): "Italie_N-S",
    #         ('27 quai de la Tournelle', 'NO-SE'): "Tournelle_NO-SE",
    #         ('27 quai de la Tournelle', 'SE-NO'): "Tournelle_SE-NO",
    #     },
    #     "sub_range": (0,),
    #     "steps_per_epoch": math.ceil(47028 / batch_size),
    #     ### Context backup
    #     "out_dir": OUT_DIR,            
    #     "context_length": context_length,
    #     "fcm_context_length": fcm_context_length,
    #     "prediction_length": prediction_length,
    #     ### Manual override for training is finished
    #     # "best_checkpoint": "checkpoint-800",
    # },
    "experiment_2": {
        "name": "Full_Counters_fcm168",
        "counters_dict": {
            ('Totem 73 boulevard de Sébastopol', 'S-N'): "Sébastopol_S-N",
            ('Totem 73 boulevard de Sébastopol', 'N-S'): "Sébastopol_N-S",
            ('102 boulevard de Magenta', 'SE-NO'): "Magenta_SE-NO",
            ('Pont de Bercy', 'NE-SO'): "Bercy_NE-SO",
            ('Pont de Bercy', 'SO-NE'): "Bercy_SO-NO",
            ('135 avenue Daumesnil', 'SE-NO'): "Daumesnil_SE-NO",
            ("180 avenue d'Italie", 'N-S'): "Italie_N-S",
            ('27 quai de la Tournelle', 'NO-SE'): "Tournelle_NO-SE",
            ('27 quai de la Tournelle', 'SE-NO'): "Tournelle_SE-NO",
            ('10 avenue de la Grande Armée', 'SE-NO'): "Armée_SE-NO",
            ('10 boulevard Auguste Blanqui', 'NE-SO'): "Blanqui_NE-SO",
            ('106 avenue Denfert Rochereau', 'NE-SO'): "Rochereau_NE-SO",
            ('129 rue Lecourbe', 'SO-NE'): "Lecourbe_SO-NE",
            ('132 rue Lecourbe', 'NE-SO'): "Lecourbe_NE-SO",
            ("147 avenue d'Italie", 'S-N'): "Italie_S-N",
            ('152 boulevard du Montparnasse', 'E-O'): "Montparnasse_E-O",
            ('152 boulevard du Montparnasse', 'O-E'): "Montparnasse_O-E",
            ('16 avenue de la Porte des Ternes', 'E-O'): "Ternes_E-O",
            ('163 boulevard Brune', 'SE-NO'): "Brune_SE-NO",
            ("18 quai de l'Hôtel de Ville", 'NO-SE'): "Ville_NO-SE",
            ("18 quai de l'Hôtel de Ville", 'SE-NO'): "Ville_SE-NO",
            ('21 boulevard Auguste Blanqui', 'SO-NE'): "Blanqui_SO-NE",
            ('24 boulevard Jourdan', 'E-O'): "Jourdan_E-O",
            ('243 boulevard Saint Germain', 'NO-SE'): "Germain_NO-SE",
            ('27 boulevard Davout', 'N-S'): "Davout_N-S",
            ('27 boulevard Diderot', 'E-O'): "Diderot_E-O",
            ('28 boulevard Diderot', 'E-O'): "Diderot_E-O",
            ('28 boulevard Diderot', 'O-E'): "Diderot_O-E",
            ('33 avenue des Champs Elysées', 'NO-SE'): "Elysées_NO-SE",
            ('35 boulevard de Ménilmontant', 'NO-SE'): "Ménilmontant_NO-SE",
            ('36 quai de Grenelle', 'NE-SO'): "Grenelle_NE-SO",
            ('36 quai de Grenelle', 'SO-NE'): "Grenelle_SO-NE",
            ('38 rue Turbigo', 'NE-SO'): "Turbigo_NE-SO",
            ('38 rue Turbigo', 'SO-NE'): "Turbigo_SO-NE",
            ('39 quai François Mauriac', 'NO-SE'): "Mauriac_NO-SE",
            ('39 quai François Mauriac', 'SE-NO'): "Mauriac_SE-NO",
            ('42 boulevard Soult', 'N-S'): "Soult_N-S",
            ('42 boulevard Soult', 'S-N'): "Soult_S-N",
            ('44 avenue des Champs Elysées', 'SE-NO'): "Elysées_SE-NO",
            ('51 boulevard du Général Martial Valin', 'SE-NO'): "Valin_SE-NO",
            ('56 boulevard Kellermann', 'E-O'): "Kellermann_E-O",
            ('6 rue Julia Bartet', 'NE-SO'): "Bartet_NE-SO",
            ('6 rue Julia Bartet', 'SO-NE'): "Bartet_SO-NE",
            ('67 boulevard Voltaire', 'SE-NO'): "Voltaire_SE-NO",
            ('7 avenue de la Grande Armée', 'NO-SE'): "Armée_NO-SE",
            ('72 avenue de Flandre', 'SO-NE'): "Flandre_SO-NE",
            ('72 boulevard Brune', 'NO-SE'): "Brune_NO-SE",
            ('72 boulevard Richard Lenoir', 'S-N'): "Lenoir_S-N",
            ('72 boulevard Voltaire', 'NO-SE'): "Voltaire_NO-SE",
            ('77 boulevard Masséna', 'NE-SO'): "Masséna_NE-SO",
            ('77 boulevard Masséna', 'SO-NE'): "Masséna_SO-NE",
            ('77 boulevard Richard Lenoir', 'N-S'): "Lenoir_N-S",
            ('81 boulevard Mortier', 'N-S'): "Mortier_N-S",
            ('81 boulevard Mortier', 'S-N'): "Mortier_S-N",
            ('87 avenue de Flandre', 'NE-SO'): "Flandre_NE-SO",
            ('89 boulevard de Magenta', 'NO-SE'): "Magenta_NO-SE",
            ('9 boulevard Jourdan', 'O-E'): "Jourdan_O-E",
            ('98 boulevard Poniatowski', 'NE-SO'): "Poniatowski_NE-SO",
            ('98 boulevard Poniatowski', 'SO-NE'): "Poniatowski_SO-NE",
            ("Face 104 rue d'Aubervilliers", 'N-S'): "Aubervilliers_N-S",
            ("Face 104 rue d'Aubervilliers", 'S-N'): "Aubervilliers_S-N",
            ('Face au 16 avenue de la  Porte des Ternes', 'O-E'): "Ternes_O-E",
            ("Face au 25 quai de l'Oise", 'NE-SO'): "Oise_NE-SO",
            ("Face au 25 quai de l'Oise", 'SO-NE'): "Oise_SO-NE",
            ('Face au 4 avenue de la porte de Bagnolet', 'E-O'): "Bagnolet_E-O",
            ('Face au 4 avenue de la porte de Bagnolet', 'O-E'): "Bagnolet_O-E",
            ("Face au 40 quai D'Issy", 'NE-SO'): "Issy_NE-SO",
            ("Face au 40 quai D'Issy", 'SO-NE'): "Issy_SO-NE",
            ('Face au 48 quai de la marne', 'NE-SO'): "Marne_NE-SO",
            ('Face au 48 quai de la marne', 'SO-NE'): "Marne_SO-NE",
            ('Face au 49 boulevard du Général Martial Valin', 'NO-SE'): "Valin_NO-SE",
            ('Face au 70 quai de Bercy', 'N-S'): "Bercy_N-S",
            ('Face au 70 quai de Bercy', 'S-N'): "Bercy_S-N",
            ('Face au 8 avenue de la porte de Charenton', 'NO-SE'): "Charenton_NO-SE",
            ('Face au 8 avenue de la porte de Charenton', 'SE-NO'): "Charenton_SE-NO",
            ('Pont Charles De Gaulle', 'NE-SO'): "Gaulle_NE-SO",
            ('Pont Charles De Gaulle', 'SO-NE'): "Gaulle_SO-NE",
            ('Pont National', 'NE-SO'): "National_NE-SO",
            ('Pont National', 'SO-NE'): "National_SO-NE",
            ('Pont de la Concorde', 'N-S'): "Concorde_N-S",
            ('Pont de la Concorde', 'S-N'): "Concorde_S-N",
            ('Pont des Invalides', 'S-N'): "Invalides_S-N",
            ('Pont des Invalides (couloir bus)', 'N-S'): "Invalides_N-S",
            ('Pont du Garigliano', 'NO-SE'): "Garigliano_NO-SE",
            ('Pont du Garigliano', 'SE-NO'): "Garigliano_SE-NO",
            ("Quai d'Orsay", 'E-O'): "Orsay_E-O",
            ("Quai d'Orsay", 'O-E'): "Orsay_O-E",
            ('Quai des Tuileries', 'NO-SE'): "Tuileries_NO-SE",
            ('Quai des Tuileries', 'SE-NO'): "Tuileries_SE-NO",
            ('Totem 64 Rue de Rivoli', 'E-O'): "Rivoli_E-O",
            ('Totem 64 Rue de Rivoli', 'O-E'): "Rivoli_O-E",
            ("Totem 85 quai d'Austerlitz", 'NO-SE'): "Austerlitz_NO-SE",
            ("Totem 85 quai d'Austerlitz", 'SE-NO'): "Austerlitz_SE-NO",
            ('Totem Cours la Reine', 'E-O'): "Reine_E-O",
            ('Totem Cours la Reine', 'O-E'): "Reine_O-E",
            ('Voie Georges Pompidou', 'NE-SO'): "Pompidou_NE-SO",
            ('Voie Georges Pompidou', 'SO-NE'): "Pompidou_SO-NE",
        },
        "sub_range": (0,),
        "steps_per_epoch": math.ceil(565532 / batch_size),
        ### Context backup
        "out_dir": OUT_DIR,            
        "context_length": context_length,
        "fcm_context_length": fcm_context_length,
        "prediction_length": prediction_length,
        ### Manual override for training is finished
        # "best_checkpoint": "checkpoint-800",
    },
}

## 3.2 Data Viz of Time Series per counter

In [ ]:
# splitted dataframe per counter
grouped_df = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

for experiment, exp_params in dict_compteurs.items():
    name = exp_params["name"]
    for key in exp_params["counters_dict"].keys():
        if key in grouped_df.groups:
            df_compteur = grouped_df.get_group(key)
            logging.info(f"\n--- {experiment} | {name} : {key} ---")
        else:
            logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
            continue

        df_compteur = df_compteur.sort_values(by=timestamp_column)
        sub_range = exp_params["sub_range"]
        range_start = sub_range[0]
        range_end = sub_range[1] if len(sub_range) > 1 else None
        df_compteur_sub = df_compteur[range_start:range_end]

        fig, axs = plt.subplots(len(target_columns), 1, figsize=(10, 2 * len(target_columns)), squeeze=False)
        for ax, target_column in zip(axs, target_columns):
            ax[0].plot(df_compteur_sub[timestamp_column], df_compteur_sub[target_column])
        plt.show()

## 3.3 Transfert learning with preprocessing and training

#### Définition et ajustement du modèle granite pour finetuning avec gel des couches pré-entrainées
> Environ 500k paramètres gelés mais il reste ceux ajoutés via notre contexte de variables exogènes, le nombre de paramètres additionnels dépend des variables exogènes et de la fenetre de contexte (FCM)
> - 24 lags (1 jour) -> environ 900k paramètres
> - 48 lags (2 jours) -> environ 2,7M paramètres
> - 168 lags (7 jours) -> environ 29M de paramètres

#### Boucle d'entrainement du modèle

In [ ]:
model_results = {}
# Make a forecast on the target column given the input data.
for experiment, exp_params in dict_compteurs.items():
    # Filtrage de l'experience et de son dataset associé
    compteur_keys = list(exp_params["counters_dict"].keys())
    grouped_df = df.groupby([
        "nom_du_site_de_comptage",
        "orientation_compteur",
    ])
    df_final = df[
        df[["nom_du_site_de_comptage", "orientation_compteur"]]
        .apply(tuple, axis=1)  # type: ignore
        .isin(compteur_keys)
    ].copy()
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_final_sub = cast(pd.DataFrame, df_final[range_start:range_end])

    # Definition et fine tuning du modèle
    name = exp_params["name"]
    steps_per_epoch = exp_params["steps_per_epoch"]
    deploy_dir = os.path.join(OUT_DIR, f"{name}_deploy")
    preproc_dir = os.path.join(OUT_DIR, f"{name}_preproc")
    output_dir = os.path.join(OUT_DIR, f"{name}_output")
    logging_dir = os.path.join(OUT_DIR, f"{name}_log")
    (
        tsp,
        model,
        args,
        optimizer,
        scheduler,
        early_stop_cb,
        tracking_cb
    ) = dlps.fine_tune_model(
        output_dir,
        logging_dir,
        context_length,
        prediction_length,
        fcm_context_length,
        column_specifiers,
        learning_rate,
        num_epochs,
        batch_size,
        steps_per_epoch,
        patience,
        torch_device.type,        
    )

    # Split train, valid, test for dataframes and datasets for training
    df_train, df_valid, df_test = prepare_data_splits(  # type: ignore
        df_final_sub,
        context_length=context_length,
        split_config=split_config  # type: ignore
    )
    logging.info(f"Dataframe lengths: train = {len(df_train)}, val = {len(df_valid)}, test = {len(df_test)}")
    dataset_train, dataset_valid, dataset_test = get_datasets(  # type: ignore
        tsp,
        df_final_sub,
        split_config,  # type: ignore
        stride=prediction_length,
        fewshot_fraction=fewshot_fraction,
        fewshot_location="first",
        use_frequency_token=model.config.resolution_prefix_tuning,
    )
    logging.info(f"Dataset batch lengths: train = {len(dataset_train)}, val = {len(dataset_valid)}, test = {len(dataset_test)}")

    # Définition du modèle et de son trainer
    finetune_forecast_trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset_train,
        eval_dataset=dataset_valid,
        callbacks=[early_stop_cb, tracking_cb],
        optimizers=(optimizer, scheduler)  # type: ignore
    )

    # Priorité au reentrainement à partir du best checkpoint
    best_checkpoint = dlps.train_or_resume(finetune_forecast_trainer, exp_params)

    # sauvegarde des résultats en mémoire
    model_results[experiment] = {
        "exp_params": exp_params,
        "df_train": df_train,
        "df_valid": df_valid,
        "df_test": df_test,
        "best_checkpoint": best_checkpoint,
        "model": model,
        "tsp": tsp,
    }

    # sauvegarde sur disque de l'experience via joblib / yaml (preprocessor inclus)
    dlps.save_granite_model(experiment, model_results[experiment])
    dlps.save_preprocessor_state(tsp, preproc_dir)
    save_deployment_package(deploy_dir, model, ts_processor=tsp)

## 3.3 Predictions

#### From memory context

In [ ]:
for experiment, model_result in model_results.items():
    # Extract the artifacts from the saved results
    exp_params = model_result["exp_params"]
    df_train = model_result["df_train"]
    df_eval = model_result["df_test"]
    model = model_result["model"]
    tsp = model_result["tsp"]
    logging.info(f"Paramètres\n{pprint.pformat(exp_params)}")
    name = exp_params["name"]
    out_dir = exp_params["out_dir"]
    checkpoint_dir = os.path.join(out_dir, f"output_{name}", model_result['best_checkpoint'])
    preproc_dir = os.path.join(out_dir, f"preproc_{name}")
    
    # Create the evaluation pipeline
    pipeline = TimeSeriesForecastingPipeline(
        model=model,
        device=torch_device,
        feature_extractor=tsp,
        batch_size=batch_size,
    )

    # Collect and store the predictions
    predictions_df_train = pipeline(df_train)  # type: ignore
    predictions_df_test = pipeline(df_eval)  # type: ignore
    model_results[experiment]["predictions_df_train"] = predictions_df_train
    model_results[experiment]["predictions_df_test"] = predictions_df_test

#### Evaluation sur les compteurs d'entrainement

In [ ]:
for experiment, model_result in model_results.items():
    # Extract the artifacts from the saved results
    exp_params = model_result["exp_params"]
    df_eval = model_result["df_train"]
    preds_df_eval = model_result["predictions_df_train"]
    logging.info(f"Paramètres\n{pprint.pformat(exp_params)}")
    # Regroup by counter
    grouped_df_eval = df_eval.groupby([
        "nom_du_site_de_comptage",
        "orientation_compteur",
    ])
    logging.info(f"Nombre de grouped_df_eval : [{len(grouped_df_eval.groups.keys())}]")
    grouped_preds_df_eval = preds_df_eval.groupby([
        "nom_du_site_de_comptage",
        "orientation_compteur",
    ])
    logging.info(f"Nombre de grouped_preds_df_eval : [{len(grouped_preds_df_eval.groups.keys())}]")
    for compteur_key, compteur_df_eval in grouped_df_eval:
        logging.info(f"Forecast à partir d la dernière date connue pour {compteur_key}")
        compteur_df_eval = grouped_df_eval.get_group(compteur_key).sort_values(by=timestamp_column).reset_index()
        compteur_preds_df_eval = grouped_preds_df_eval.get_group(compteur_key).sort_values(by=timestamp_column).reset_index()
        plot_predictions(
            input_df=compteur_df_eval,
            predictions_df=compteur_preds_df_eval,  # type: ignore
            freq="h",
            timestamp_column=timestamp_column,
            channel=target_columns[0],
            # we check the prediction in the future (from the last index)
            indices=[-1],
            num_plots=4,
        )
        plt.show()

        # split simple pour vérifier sur les même données que nos autres modèles
        compteur_preds_df_eval, compteur_preds_df_test = dlps.df_split_time_aware(
            compteur_preds_df_eval,
            timestamp_column=timestamp_column,
            test_size=0.25,
            sort=True,
        )    
        y_train = compteur_preds_df_eval.comptage_horaire.apply(
            lambda x: x[0]
        )[:-1].fillna(0)
        y_train_pred = compteur_preds_df_eval.comptage_horaire_prediction.apply(
            lambda x: x[0]
        )[:-1].fillna(0)
        y_test = compteur_preds_df_test.comptage_horaire.apply(
            lambda x: x[0]
        )[:-1].fillna(0)
        y_test_pred = compteur_preds_df_test.comptage_horaire_prediction.apply(
            lambda x: x[0]
        )[:-1].fillna(0)
        dates_test = compteur_preds_df_test[["date_et_heure_de_comptage_local"]][:-1]

        periode_limite = (
            dates_test.date_et_heure_de_comptage_local[max(dates_test.index.max()-24*7*4,
                                                           dates_test.index.min())],
            dates_test.date_et_heure_de_comptage_local[dates_test.index.max()-1],
        )
        # Affichage des metrique train et test
        model_train_metrics = mps.compute_metrics(
            y_train,
            y_train_pred
        )
        model_test_metrics = mps.compute_metrics(
            y_test,
            y_test_pred
        )
        logging.info(f"Metriques du modèle (Train): {model_train_metrics}")
        logging.info(f"Metriques du modèle (Test): {model_test_metrics}")

        # projection des predictions de test dans le temps
        fig_pred = mps.plot_predictions(
            str(compteur_key),
            dates_test, 
            y_test, 
            y_test_pred, 
            periode_limite=periode_limite,  # type: ignore
        )
        plt.show()

        # projection des résidus et calcul du coefficient de dérive dans le temps
        fig1_res, fig2_res, model_res_coeff = mps.compute_residuals_plot(
            str(compteur_key),
            dates_test, 
            y_test, 
            y_test_pred, 
            periode_limite=periode_limite  # type: ignore
        )
        logging.info(f"Pente de la droite de régression des résidus dans le temps (dérive) : {model_res_coeff}")
        plt.show()

#### Evaluation sur les compteurs de test différents de ceux d'entrainement

In [ ]:
for experiment, model_result in model_results.items():
    # Extract the artifacts from the saved results
    exp_params = model_result["exp_params"]
    df_eval = model_result["df_test"]
    preds_df_eval = model_result["predictions_df_test"]
    logging.info(f"Paramètres\n{pprint.pformat(exp_params)}")
    # Regroup by counter
    grouped_df_eval = df_eval.groupby([
        "nom_du_site_de_comptage",
        "orientation_compteur",
    ])
    logging.info(f"Nombre de grouped_df_eval : [{len(grouped_df_eval.groups.keys())}]")
    grouped_preds_df_eval = preds_df_eval.groupby([
        "nom_du_site_de_comptage",
        "orientation_compteur",
    ])
    logging.info(f"Nombre de grouped_preds_df_eval : [{len(grouped_preds_df_eval.groups.keys())}]")
    for compteur_key, compteur_df_eval in grouped_df_eval:
        logging.info(f"Forecast à partir d la dernière date connue pour {compteur_key}")
        compteur_df_eval = grouped_df_eval.get_group(compteur_key).sort_values(by=timestamp_column).reset_index()
        compteur_preds_df_eval = grouped_preds_df_eval.get_group(compteur_key).sort_values(by=timestamp_column).reset_index()
        plot_predictions(
            input_df=compteur_df_eval,
            predictions_df=compteur_preds_df_eval,  # type: ignore
            freq="h",
            timestamp_column=timestamp_column,
            channel=target_columns[0],
            # we check the prediction in the future (from the last index)
            indices=[-1],
            num_plots=4,
        )
        plt.show()

        # split simple pour vérifier sur les même données que nos autres modèles
        compteur_preds_df_train, compteur_preds_df_test = dlps.df_split_time_aware(
            compteur_preds_df_eval,
            timestamp_column=timestamp_column,
            test_size=0.25,
            sort=True,
        )    
        y_train = compteur_preds_df_train.comptage_horaire.apply(
            lambda x: x[0]
        )[:-1].fillna(0)
        y_train_pred = compteur_preds_df_train.comptage_horaire_prediction.apply(
            lambda x: x[0]
        )[:-1].fillna(0)
        y_test = compteur_preds_df_test.comptage_horaire.apply(
            lambda x: x[0]
        )[:-1].fillna(0)
        y_test_pred = compteur_preds_df_test.comptage_horaire_prediction.apply(
            lambda x: x[0]
        )[:-1].fillna(0)
        dates_test = compteur_preds_df_test[["date_et_heure_de_comptage_local"]][:-1]
        periode_limite = (
            dates_test.date_et_heure_de_comptage_local[max(dates_test.index.max()-24*7*4,
                                                           dates_test.index.min())],
            dates_test.date_et_heure_de_comptage_local[dates_test.index.max()-1],
        )
        # Affichage des metrique train et test
        model_train_metrics = mps.compute_metrics(
            y_train,
            y_train_pred
        )
        model_test_metrics = mps.compute_metrics(
            y_test,
            y_test_pred
        )
        logging.info(f"Metriques du modèle (Train): {model_train_metrics}")
        logging.info(f"Metriques du modèle (Test): {model_test_metrics}")

        # projection des predictions de test dans le temps
        fig_pred = mps.plot_predictions(
            str(compteur_key),
            dates_test, 
            y_test, 
            y_test_pred, 
            periode_limite=periode_limite,  # type: ignore
        )
        plt.show()

        # projection des résidus et calcul du coefficient de dérive dans le temps
        fig1_res, fig2_res, model_res_coeff = mps.compute_residuals_plot(
            str(compteur_key),
            dates_test, 
            y_test, 
            y_test_pred, 
            periode_limite=periode_limite  # type: ignore
        )
        logging.info(f"Pente de la droite de régression des résidus dans le temps (dérive) : {model_res_coeff}")
        plt.show()